In [0]:

# =========================================================
# PROJECT: VIDEO GAME SALES DATA PLATFORM
# PHASE 2: BRONZE LAYER - RAW DATA INGESTION
# UNITY CATALOG COMPATIBLE
# =========================================================

from pyspark.sql.functions import current_timestamp, col

# 1. Source and destination
source_path = (
    "/Volumes/workspace/video_game_sales/"
    "raw_files/video_game_sales.csv"
)

bronze_table = (
    "workspace.video_game_sales.bronze_video_game_sales"
)

# 2. Read source data
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

source_count = raw_df.count()

print("Source record count:", source_count)

# 3. Read file metadata using Unity Catalog-compatible approach
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
    .select(
        "*",
        col("_metadata.file_path").alias("source_file")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

# 4. Write Bronze Delta table
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print("Bronze table created:", bronze_table)

# 5. Validate record count
bronze_count = spark.table(bronze_table).count()

print("Bronze record count:", bronze_count)

# 6. Reconciliation
assert source_count == bronze_count, (
    "Bronze ingestion reconciliation failed"
)

print("BRONZE INGESTION VALIDATION PASSED")


In [0]:
spark.sql("""
DESCRIBE TABLE workspace.video_game_sales.bronze_video_game_sales
""").show(truncate=False)

display(
    spark.table(
        "workspace.video_game_sales.bronze_video_game_sales"
    ).limit(10)
)

In [0]:
bronze_df = spark.table(
    "workspace.video_game_sales.bronze_video_game_sales"
)

print("Bronze records:", bronze_df.count())

print("Source records:", raw_df.count())

print("Distinct source files:",
      bronze_df.select("source_file").distinct().count())

print("Null ingestion timestamps:",
      bronze_df.filter(
          "ingestion_timestamp IS NULL"
      ).count())

assert bronze_df.count() == 16598
assert bronze_df.filter(
    "ingestion_timestamp IS NULL"
).count() == 0

print("BRONZE VALIDATION PASSED")